# Artist Demand Analysis

## Purpose

Test whether unusual artist supply-side behaviour is accompanied by demand-side movement.

The analysis evaluates:

1. Temporal artist demand across multiple horizons
2. Track-level demand concentration and breadth
3. Whether demand is concentrated in one/few tracks or distributed more broadly
4. Connection between track demand and catalogue/release context where supported

Maddix is used as the initial validation case.

### Evidence limitations

ListenBrainz artist-level temporal statistics and track-level popularity statistics represent different analytical views and should not be treated as directly equivalent.

Cumulative track popularity must not be interpreted as recent track-level momentum unless temporal track evidence is available.

In [1]:
import os
import time
import requests
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

LISTENBRAINZ_TOKEN = os.getenv("LISTENBRAINZ_TOKEN")

if not LISTENBRAINZ_TOKEN:
    raise ValueError("LISTENBRAINZ_TOKEN not found in .env")

HEADERS = {
    "Authorization": f"Token {LISTENBRAINZ_TOKEN}"
}

MADDIX_MBID = "0ae6eb0e-f5b4-48b0-98ab-2158ccd6f5c8"

print("ListenBrainz token loaded:", bool(LISTENBRAINZ_TOKEN))
print("Maddix MBID:", MADDIX_MBID)

ListenBrainz token loaded: True
Maddix MBID: 0ae6eb0e-f5b4-48b0-98ab-2158ccd6f5c8


In [2]:
def fetch_artist_temporal_demand(artist_mbid):
    ranges = ["week", "month", "quarter", "half_yearly", "year"]
    rows = []

    url = (
        "https://api.listenbrainz.org/1/stats/artist/"
        f"{artist_mbid}/listeners"
    )

    for period in ranges:
        print(f"Fetching {period}...")

        for attempt in range(3):
            try:
                response = requests.get(
                    url,
                    params={"range": period},
                    timeout=60
                )
                response.raise_for_status()

                payload = response.json()["payload"]

                rows.append({
                    "range": period,
                    "total_listens": payload["total_listen_count"],
                    "unique_listeners": payload["total_user_count"],
                    "listens_per_listener":
                        payload["total_listen_count"]
                        / payload["total_user_count"]
                        if payload["total_user_count"] else None,
                    "last_updated":
                        pd.to_datetime(payload["last_updated"], unit="s")
                })

                break

            except requests.exceptions.RequestException as e:
                print(
                    f"Attempt {attempt + 1} failed: "
                    f"{type(e).__name__}"
                )

                if attempt == 2:
                    print(f"Skipping {period}.")
                else:
                    time.sleep(5)

        time.sleep(2)

    return pd.DataFrame(rows)

In [3]:
maddix_temporal_demand = fetch_artist_temporal_demand(MADDIX_MBID)

maddix_temporal_demand

Fetching week...
Fetching month...
Fetching quarter...
Fetching half_yearly...
Fetching year...


,range,total_listens,unique_listeners,listens_per_listener,last_updated
0,week,178,81,2.197531,2026-10-03 05:03:11
1,month,671,207,3.241546,2026-10-03 06:18:31
2,quarter,2142,482,4.443983,2026-10-03 08:35:23
3,half_yearly,4581,734,6.241144,2026-10-03 15:48:57
4,year,9701,1081,8.974098,2026-10-03 12:20:56


## 1. Temporal Artist Demand

Normalize overlapping ListenBrainz demand windows to provide context on recent listening intensity relative to longer-term windows.

These measures describe demand intensity, not growth rates, because the observation windows overlap.

In [4]:
period_days = {
    "week": 7,
    "month": 30,
    "quarter": 90,
    "half_yearly": 180,
    "year": 365
}

maddix_temporal_demand["days"] = (
    maddix_temporal_demand["range"].map(period_days)
)

maddix_temporal_demand["listens_per_day"] = (
    maddix_temporal_demand["total_listens"]
    / maddix_temporal_demand["days"]
)

maddix_temporal_demand["listeners_per_day"] = (
    maddix_temporal_demand["unique_listeners"]
    / maddix_temporal_demand["days"]
)

maddix_temporal_view = maddix_temporal_demand[
    [
        "range",
        "total_listens",
        "unique_listeners",
        "listens_per_listener",
        "listens_per_day",
        "listeners_per_day"
    ]
].round(2)

maddix_temporal_view

,range,total_listens,unique_listeners,listens_per_listener,listens_per_day,listeners_per_day
0,week,178,81,2.20,25.43,11.57
1,month,671,207,3.24,22.37,6.90
2,quarter,2142,482,4.44,23.80,5.36
3,half_yearly,4581,734,6.24,25.45,4.08
4,year,9701,1081,8.97,26.58,2.96


## 2. Track-Level Demand and Breadth

Retrieve ListenBrainz popularity statistics for Maddix recordings.

This provides track-level consumption decomposition and is used to evaluate whether observed demand is concentrated in a small number of recordings or distributed across a broader set.

Important: these statistics are not treated as temporal track momentum unless the endpoint explicitly provides a matching temporal observation window.

In [9]:
def fetch_artist_track_demand(artist_mbid, count=100):
    url = (
        "https://api.listenbrainz.org/1/popularity/"
        f"top-recordings-for-artist/{artist_mbid}"
    )

    response = requests.get(
        url,
        headers=HEADERS,
        params={"count": count},
        timeout=60
    )

    response.raise_for_status()

    # This endpoint returns the recordings directly as a list
    recordings = response.json()

    rows = []

    for item in recordings:
        rows.append({
            "recording_name": item.get("recording_name"),
            "release_name": item.get("release_name"),
            "recording_mbid": item.get("recording_mbid"),
            "release_mbid": item.get("release_mbid"),
            "total_listens": item.get("total_listen_count"),
            "total_users": item.get("total_user_count")
        })

    return pd.DataFrame(rows)

In [10]:
maddix_tracks = fetch_artist_track_demand(
    MADDIX_MBID,
    count=100
)

print("Tracks returned:", len(maddix_tracks))

maddix_tracks.head(20)

Tracks returned: 209


,recording_name,release_name,recording_mbid,release_mbid,total_listens,total_users
0,Heute Nacht,Heute Nacht,abd5d9f1-fb35-4d1a-8dcd-8fc492ed5b64,4ee044a8-062c-4e9e-b54f-499c1e97b3e7,5642,956
1,My Gasoline,My Gasoline,a193f03b-1aeb-4b46-9b56-f92168b419ed,6ae78105-8b11-4198-9581-bbd8e9a597d4,3537,732
2,Bella ciao,Bella Ciao,82c301c9-23bf-482b-9353-243f5a2e9582,14b2646e-23fa-4135-ab7c-05a3b6e520bc,2397,511
3,Activating,NaN,7b4734c7-97f5-4847-943c-28e337bddf76,NaN,2144,457
4,90s Bitch,90s Bitch,7498d0a2-d78c-451e-b59c-4993ef225f11,9200f967-d222-43ab-9faf-e29251f00eaa,1741,395
5,The Prophecy,The Prophecy,1bcab57a-7af4-4ca3-9ecd-1e1876e89743,ec82b6b4-6aef-48e8-9390-27bb71a070e3,1576,259
6,ACID,Acid,a24d2852-91d0-43cd-bfa1-7445d5f1f18e,adf56ff0-5c7e-4cff-bad4-eb2386e44c4d,1479,319
7,Revolution,Revolution,28aeeccd-2a43-42bb-92aa-7dd62c75b305,df1ecbde-c9d5-49e8-8a16-271720c18945,1252,309
8,Superheroes,Superheroes,8a1fe3f3-a57a-4013-9d9f-66ee8c1c14d9,9333920c-77c3-4185-aa33-39e0da283da2,1101,220
9,Meet Her at the Love Parade,Meet Her At The Love Parade,626c8d37-5258-4ba0-903a-78855bb56471,e18e157a-7372-48cc-801c-f41829239239,1050,318


### Track Demand Concentration

Measure how cumulative ListenBrainz consumption is distributed across Maddix recordings.

Top-1, Top-5 and Top-10 shares provide an interpretable view of whether observed demand is dominated by a small number of recordings or distributed more broadly.

These are cumulative popularity measures and should not be interpreted as recent track-level momentum.

In [11]:
maddix_track_breadth = (
    maddix_tracks
    .drop_duplicates(subset=["recording_mbid"])
    .sort_values("total_listens", ascending=False)
    .reset_index(drop=True)
)

total_track_listens = maddix_track_breadth["total_listens"].sum()

maddix_track_breadth["listen_share_pct"] = (
    maddix_track_breadth["total_listens"]
    / total_track_listens * 100
)

maddix_track_breadth["cumulative_share_pct"] = (
    maddix_track_breadth["listen_share_pct"].cumsum()
)

concentration_summary = pd.DataFrame({
    "metric": [
        "Tracks represented",
        "Total listens represented",
        "Top 1 share (%)",
        "Top 5 share (%)",
        "Top 10 share (%)"
    ],
    "value": [
        len(maddix_track_breadth),
        total_track_listens,
        maddix_track_breadth["listen_share_pct"].iloc[0],
        maddix_track_breadth["listen_share_pct"].head(5).sum(),
        maddix_track_breadth["listen_share_pct"].head(10).sum()
    ]
})

concentration_summary.round(2)

,metric,value
0,Tracks represented,207.00
1,Total listens represented,44903.00
2,Top 1 share (%),12.56
3,Top 5 share (%),34.43
4,Top 10 share (%),48.81


In [12]:
maddix_track_breadth[
    [
        "recording_name",
        "release_name",
        "total_listens",
        "total_users",
        "listen_share_pct",
        "cumulative_share_pct"
    ]
].head(20).round(2)

,recording_name,release_name,total_listens,total_users,listen_share_pct,cumulative_share_pct
0,Heute Nacht,Heute Nacht,5642,956,12.56,12.56
1,My Gasoline,My Gasoline,3537,732,7.88,20.44
2,Bella ciao,Bella Ciao,2397,511,5.34,25.78
3,Activating,NaN,2144,457,4.77,30.55
4,90s Bitch,90s Bitch,1741,395,3.88,34.43
5,The Prophecy,The Prophecy,1576,259,3.51,37.94
6,ACID,Acid,1479,319,3.29,41.24
7,Revolution,Revolution,1252,309,2.79,44.02
8,Superheroes,Superheroes,1101,220,2.45,46.48
9,Meet Her at the Love Parade,Meet Her At The Love Parade,1050,318,2.34,48.81


In [13]:
from pathlib import Path

SILVER_PATH = Path("../data/silver/musicbrainz")

maddix_catalogue = pd.read_csv(
    SILVER_PATH / "maddix_release_catalogue.csv"
)

maddix_catalogue["first_release_date"] = pd.to_datetime(
    maddix_catalogue["first_release_date"]
)

print("Catalogue rows:", len(maddix_catalogue))
print(maddix_catalogue.columns.tolist())

maddix_catalogue.head()

Catalogue rows: 99
['artist_name', 'release_group_id', 'title', 'primary_type', 'first_release_date', 'is_compilation', 'is_remix', 'is_live', 'is_dj_mix']


,artist_name,release_group_id,title,primary_type,first_release_date,is_compilation,is_remix,is_live,is_dj_mix
0,Maddix,3a0e0471-7cbb-4bab-848c-f6d18ca8319f,True Blood,Single,2015-05-30,False,False,False,False
1,Maddix,f3944d88-55ea-47fa-aa28-1deb0b5dc77f,Riptide,Single,2015-09-04,False,False,False,False
2,Maddix,fd4c0988-51d8-4b1d-b3a8-1890382c4ab6,Ghosts,Single,2015-11-30,False,False,False,False
3,Maddix,5d88c7b1-fc30-4118-83bd-83c98a909b84,Voltage,Single,2016-05-27,False,False,False,False
4,Maddix,56af2d2d-40b3-4bc0-9672-e1c568840003,Dirty Bassline,Single,2016-07-25,False,False,False,False


## 3. Demand × Catalogue Context

Connect ListenBrainz track demand to the existing MusicBrainz catalogue to evaluate whether observed consumption is associated with recent releases or the broader catalogue.

The current datasets expose different MusicBrainz entity identifiers, so title-based matching is tested first and match coverage is explicitly measured before using the result analytically.

In [14]:
import re
import unicodedata

def normalize_title(value):
    if pd.isna(value):
        return None

    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.lower().strip()

    # Standardize punctuation/spacing
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


maddix_track_breadth["title_key"] = (
    maddix_track_breadth["release_name"]
    .fillna(maddix_track_breadth["recording_name"])
    .apply(normalize_title)
)

maddix_catalogue["title_key"] = (
    maddix_catalogue["title"].apply(normalize_title)
)

In [15]:
catalogue_title_duplicates = (
    maddix_catalogue
    .groupby("title_key")
    .size()
    .reset_index(name="catalogue_records")
    .query("catalogue_records > 1")
    .sort_values("catalogue_records", ascending=False)
)

print(
    "Duplicate normalized catalogue titles:",
    len(catalogue_title_duplicates)
)

catalogue_title_duplicates.head(20)

Duplicate normalized catalogue titles: 0


,title_key,catalogue_records


In [16]:
maddix_demand_catalogue = (
    maddix_track_breadth
    .merge(
        maddix_catalogue[
            [
                "title_key",
                "title",
                "first_release_date",
                "primary_type"
            ]
        ],
        on="title_key",
        how="left",
        indicator=True
    )
)

maddix_demand_catalogue.head(20)

,recording_name,release_name,recording_mbid,release_mbid,total_listens,total_users,listen_share_pct,cumulative_share_pct,title_key,title,first_release_date,primary_type,_merge
0,Heute Nacht,Heute Nacht,abd5d9f1-fb35-4d1a-8dcd-8fc492ed5b64,4ee044a8-062c-4e9e-b54f-499c1e97b3e7,5642,956,12.564862,12.564862,heute nacht,Heute Nacht,2022-10-14,Single,both
1,My Gasoline,My Gasoline,a193f03b-1aeb-4b46-9b56-f92168b419ed,6ae78105-8b11-4198-9581-bbd8e9a597d4,3537,732,7.876979,20.441841,my gasoline,My Gasoline,2023-08-18,Single,both
2,Bella ciao,Bella Ciao,82c301c9-23bf-482b-9353-243f5a2e9582,14b2646e-23fa-4135-ab7c-05a3b6e520bc,2397,511,5.338173,25.780015,bella ciao,Bella Ciao,2018-08-17,Single,both
3,Activating,NaN,7b4734c7-97f5-4847-943c-28e337bddf76,NaN,2144,457,4.774737,30.554751,activating,Activating,2021-01-15,Single,both
4,90s Bitch,90s Bitch,7498d0a2-d78c-451e-b59c-4993ef225f11,9200f967-d222-43ab-9faf-e29251f00eaa,1741,395,3.877247,34.431998,90s bitch,90s Bitch,2023-12-08,Single,both
5,The Prophecy,The Prophecy,1bcab57a-7af4-4ca3-9ecd-1e1876e89743,ec82b6b4-6aef-48e8-9390-27bb71a070e3,1576,259,3.509788,37.941786,the prophecy,The Prophecy,2018-09-28,Single,both
6,ACID,Acid,a24d2852-91d0-43cd-bfa1-7445d5f1f18e,adf56ff0-5c7e-4cff-bad4-eb2386e44c4d,1479,319,3.293767,41.235552,acid,ACID,2023-05-05,Single,both
7,Revolution,Revolution,28aeeccd-2a43-42bb-92aa-7dd62c75b305,df1ecbde-c9d5-49e8-8a16-271720c18945,1252,309,2.788232,44.023785,revolution,Revolution,2023-03-17,Single,both
8,Superheroes,Superheroes,8a1fe3f3-a57a-4013-9d9f-66ee8c1c14d9,9333920c-77c3-4185-aa33-39e0da283da2,1101,220,2.451952,46.475737,superheroes,Superheroes,2021-02-05,Single,both
9,Meet Her at the Love Parade,Meet Her At The Love Parade,626c8d37-5258-4ba0-903a-78855bb56471,e18e157a-7372-48cc-801c-f41829239239,1050,318,2.338374,48.814110,meet her at the love parade,Meet Her At The Love Parade,2024-05-17,Single,both


In [17]:
total_tracks = len(maddix_demand_catalogue)

matched_tracks = (
    maddix_demand_catalogue["_merge"] == "both"
).sum()

total_listens = maddix_demand_catalogue["total_listens"].sum()

matched_listens = maddix_demand_catalogue.loc[
    maddix_demand_catalogue["_merge"] == "both",
    "total_listens"
].sum()

match_summary = pd.DataFrame({
    "metric": [
        "Demand tracks",
        "Matched tracks",
        "Track match rate (%)",
        "Total listens",
        "Matched listens",
        "Listen coverage (%)"
    ],
    "value": [
        total_tracks,
        matched_tracks,
        matched_tracks / total_tracks * 100,
        total_listens,
        matched_listens,
        matched_listens / total_listens * 100
    ]
})

match_summary.round(2)

,metric,value
0,Demand tracks,207.00
1,Matched tracks,156.00
2,Track match rate (%),75.36
3,Total listens,44903.00
4,Matched listens,41724.00
5,Listen coverage (%),92.92


In [18]:
analysis_date = pd.Timestamp("2026-10-03")

matched = maddix_demand_catalogue[
    maddix_demand_catalogue["_merge"] == "both"
].copy()

matched["first_release_date"] = pd.to_datetime(
    matched["first_release_date"]
)

matched["release_age_days"] = (
    analysis_date - matched["first_release_date"]
).dt.days

matched["catalogue_age"] = pd.cut(
    matched["release_age_days"],
    bins=[-1, 365, 365 * 3, float("inf")],
    labels=[
        "Recent (≤1y)",
        "Mid catalogue (1–3y)",
        "Older catalogue (>3y)"
    ]
)

age_summary = (
    matched
    .groupby("catalogue_age", observed=True)
    .agg(
        tracks=("recording_name", "count"),
        listens=("total_listens", "sum"),
        listeners=("total_users", "sum")
    )
    .reset_index()
)

age_summary["listen_share_pct"] = (
    age_summary["listens"] /
    age_summary["listens"].sum() * 100
)

age_summary.round(2)

,catalogue_age,tracks,listens,listeners,listen_share_pct
0,Recent (≤1y),11,1138,504,2.73
1,Mid catalogue (1–3y),32,9050,2684,21.69
2,Older catalogue (>3y),113,31536,6982,75.58


In [19]:
import requests
import pandas as pd

ranges = ["week", "month", "quarter", "half_yearly", "year"]

results = []

for r in ranges:
    response = requests.get(
        f"https://api.listenbrainz.org/1/popularity/top-recordings-for-artist/{MADDIX_MBID}",
        headers=HEADERS,
        params={
            "count": 100,
            "range": r
        },
        timeout=60
    )

    print(r, response.status_code, response.url)

    if response.status_code == 200:
        payload = response.json()

        # endpoint may return the list directly
        if isinstance(payload, list):
            recordings = payload
        else:
            recordings = payload.get("payload", payload)

        results.append({
            "range": r,
            "recordings_returned": len(recordings),
            "top_track": recordings[0].get("recording_name") if recordings else None,
            "top_track_listens": recordings[0].get("total_listen_count") if recordings else None
        })

range_test = pd.DataFrame(results)
range_test

week 200 https://api.listenbrainz.org/1/popularity/top-recordings-for-artist/0ae6eb0e-f5b4-48b0-98ab-2158ccd6f5c8?count=100&range=week
month 200 https://api.listenbrainz.org/1/popularity/top-recordings-for-artist/0ae6eb0e-f5b4-48b0-98ab-2158ccd6f5c8?count=100&range=month
quarter 200 https://api.listenbrainz.org/1/popularity/top-recordings-for-artist/0ae6eb0e-f5b4-48b0-98ab-2158ccd6f5c8?count=100&range=quarter
half_yearly 200 https://api.listenbrainz.org/1/popularity/top-recordings-for-artist/0ae6eb0e-f5b4-48b0-98ab-2158ccd6f5c8?count=100&range=half_yearly
year 200 https://api.listenbrainz.org/1/popularity/top-recordings-for-artist/0ae6eb0e-f5b4-48b0-98ab-2158ccd6f5c8?count=100&range=year


,range,recordings_returned,top_track,top_track_listens
0,week,209,Heute Nacht,5642
1,month,209,Heute Nacht,5642
2,quarter,209,Heute Nacht,5642
3,half_yearly,209,Heute Nacht,5642
4,year,209,Heute Nacht,5642


## 4. Temporal Track Demand Feasibility

Test whether ListenBrainz sitewide recording statistics provide
sufficient temporal track-level coverage for Maddix.

This is a feasibility test, not yet part of the production pipeline.

In [20]:
import requests
import pandas as pd

ranges = ["week", "month", "quarter", "half_yearly", "year"]

rows = []

for r in ranges:
    print(f"Fetching {r}...")

    response = requests.get(
        "https://api.listenbrainz.org/1/stats/sitewide/recordings",
        params={
            "range": r,
            "count": 1000
        },
        timeout=60
    )

    print("Status:", response.status_code)

    if response.status_code != 200:
        continue

    payload = response.json()["payload"]
    recordings = payload.get("recordings", [])

    for item in recordings:
        artist_mbids = item.get("artist_mbids") or []

        if MADDIX_MBID in artist_mbids:
            rows.append({
                "range": r,
                "track_name": item.get("track_name"),
                "release_name": item.get("release_name"),
                "recording_mbid": item.get("recording_mbid"),
                "listen_count": item.get("listen_count"),
                "from_ts": payload.get("from_ts"),
                "to_ts": payload.get("to_ts")
            })

maddix_temporal_tracks = pd.DataFrame(rows)

maddix_temporal_tracks

Fetching week...
Status: 200
Fetching month...
Status: 200
Fetching quarter...
Status: 200
Fetching half_yearly...
Status: 200
Fetching year...
Status: 200


""


In [21]:
if not maddix_temporal_tracks.empty:
    temporal_coverage = (
        maddix_temporal_tracks
        .groupby("range")
        .agg(
            tracks=("recording_mbid", "nunique"),
            listens=("listen_count", "sum")
        )
        .reset_index()
    )

    temporal_coverage
else:
    print("No Maddix recordings found in sitewide top-recording statistics.")

No Maddix recordings found in sitewide top-recording statistics.
